# AMR-HUB door activity

Door Message counts per rostered shift and per hour, time and activity between consecutive door interactions, and door-to-door transitions. A Door Message is treated as a door interaction, not a verified physical pass.

## Settings

Set `DB_NAME` if `/data` contains more than one database file.

In [ ]:
from pathlib import Path
from IPython.display import display, SVG

DB_NAME = None
OUT_DIR = "/outputs/door_exploration"
DEDUPE_SECONDS = 0.0

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print("Output folder:", OUT_DIR)

## Load helper

In [ ]:
import sys
from pathlib import Path

# JupyterLab is rooted at /outputs in this image.
for p in [str(Path.cwd()), "/outputs"]:
    if p not in sys.path:
        sys.path.insert(0, p)

import door_exploration_existing_jupyter as dx  # noqa: E402

print("Loaded:", dx.__file__)

## Run

Writes tables and figures to `/outputs/door_exploration`.

In [ ]:
results = dx.run_exploration(
    db=DB_NAME,
    out=OUT_DIR,
    dedupe_seconds=DEDUPE_SECONDS,
)

display(results["summary"])

## Shifts

In [ ]:
display(results["shifts"].head(20))

print("Door interactions per shift: quantiles")
display(
    results["shifts"]["door_interactions"]
    .quantile([0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1])
    .to_frame("door_interactions")
)

print("Door interactions per hour: quantiles")
display(
    results["shifts"]["door_interactions_per_hour"]
    .quantile([0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1])
    .to_frame("door_interactions_per_hour")
)

## Door interactions per shift

In [ ]:
display(SVG(filename=str(results["figures"]["01"])))

## Door interactions per rostered hour

In [ ]:
display(SVG(filename=str(results["figures"]["02"])))

## Door interactions vs shift duration

In [ ]:
display(SVG(filename=str(results["figures"]["03"])))

## Door-to-door episodes

Interval between consecutive door interactions in the same shift, with the number of non-door events in between.

In [ ]:
display(results["episodes"].head(20))

if len(results["episodes"]):
    print("Minutes between doors: quantiles")
    display(
        results["episodes"]["duration_min"]
        .quantile([0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1])
        .to_frame("duration_min")
    )
    print("Activities between doors: quantiles")
    display(
        results["episodes"]["events_between"]
        .quantile([0, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1])
        .to_frame("events_between")
    )

## Minutes between door interactions

In [ ]:
display(SVG(filename=str(results["figures"]["04"])))

## Activities between door interactions

In [ ]:
display(SVG(filename=str(results["figures"]["05"])))

## Door transition matrix

Rows: current door. Columns: next door. Sequence order only, not direction through a door.

In [ ]:
display(SVG(filename=str(results["figures"]["06"])))

## Most-used doors and transitions

In [ ]:
print("Most-used doors")
display(results["door_counts"].head(20))

if not results["transition_matrix"].empty:
    long = (
        results["transition_matrix"]
        .stack()
        .reset_index(name="episodes")
        .query("episodes > 0")
        .sort_values("episodes", ascending=False)
    )
    display(long.head(25))

## Output files

In [ ]:
print("Folder:", results["out_dir"])
print("\nCSV files:")
for p in sorted(Path(results["out_dir"]).glob("*.csv")):
    print(" -", p.name)
print("\nFigures:")
for p in sorted(Path(results["out_dir"]).glob("*.svg")):
    print(" -", p.name)